<div style='background: linear-gradient(135deg, #1a2f5e 0%, #2c5282 100%); padding: 32px 40px; border-radius: 8px;'>
  <p style='color: #90afd4; margin: 0 0 8px 0; font-size: 0.75em; letter-spacing: 2px; text-transform: uppercase;'>Module [X] · Portfolio Risk Estimation</p>
  <h1 style='color: #ffffff; font-size: 2em; font-weight: 700; margin: 0 0 8px 0;'>[Estimator Name]</h1>
  <p style='color: #a8c4e0; font-size: 0.95em; font-weight: 300; margin: 0;'>Author: [Your Name]</p>
</div>

---
## §1 Motivation

*What problem does this estimator solve? No equations yet — just the intuition.*

[Write ~200 words explaining why this estimator exists. What's wrong with the sample covariance that this fixes? What assumption does it make? When would you expect it to work well?]

---
## §2 Derivation

*Derive the estimator from first principles. Show the math.*

### Starting point

[Begin with the problem setup. What are we trying to estimate? What loss function are we minimizing?]

$$\text{[Your key equation here]}$$

### Main result

[Work through the derivation step by step. End with the closed-form or algorithmic solution.]

$$\hat{\boldsymbol{\Sigma}} = \text{[Your estimator]}$$

---
## §3 Implementation

In [ ]:
# ---------------------------------------------------------
# Setup: Install dependencies & load data
# ---------------------------------------------------------
%pip install -q scipy yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import os
import warnings
warnings.filterwarnings('ignore')

# Load universe data — fallback to download if CSVs don't exist
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(f'{DATA_DIR}/prices.csv') and os.path.exists(f'{DATA_DIR}/returns.csv'):
    prices = pd.read_csv(f'{DATA_DIR}/prices.csv', index_col=0, parse_dates=True)
    returns = pd.read_csv(f'{DATA_DIR}/returns.csv', index_col=0, parse_dates=True)
    print("Loaded data from CSV files")
else:
    print("CSV files not found — downloading from yfinance...")
    import yfinance as yf
    TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'JNJ', 'XOM', 'PG', 'UNH', 'V', 
               'HD', 'MA', 'PFE', 'KO', 'PEP', 'MRK', 'ABBV', 'CVX', 'LLY', 'COST']
    prices = yf.download(TICKERS, start='2018-01-01', end='2024-12-31', auto_adjust=True)['Close']
    prices = prices.dropna()
    returns = prices.pct_change().dropna()
    # Save for next time
    prices.to_csv(f'{DATA_DIR}/prices.csv')
    returns.to_csv(f'{DATA_DIR}/returns.csv')
    print("Data downloaded and saved to CSV")

print(f"\nLoaded: {returns.shape[0]} days, {returns.shape[1]} assets")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ---------------------------------------------------------
# YOUR ESTIMATOR: Implement here
# ---------------------------------------------------------

def estimate_covariance(returns_df):
    """
    Estimate the covariance matrix using [YOUR METHOD].
    
    Parameters
    ----------
    returns_df : pd.DataFrame
        T x N dataframe of asset returns
        
    Returns
    -------
    cov : np.ndarray
        N x N covariance matrix estimate
    """
    # TODO: Replace with your estimator
    # This is the sample covariance as placeholder
    return returns_df.cov().values


# Quick sanity check
cov_test = estimate_covariance(returns.iloc[:100])
assert cov_test.shape == (returns.shape[1], returns.shape[1]), "Shape mismatch"
assert np.allclose(cov_test, cov_test.T), "Not symmetric"
assert np.all(np.linalg.eigvalsh(cov_test) > -1e-10), "Not positive semi-definite"
print("✓ Estimator passes basic checks")

---
## §4 Backtest

We run a rolling monthly backtest comparing three strategies:
1. **Equal-weight** — $w_i = 1/N$ (no estimation needed)
2. **Sample covariance MVO** — standard baseline
3. **Your estimator MVO** — the method you implemented

In [ ]:
# ---------------------------------------------------------
# Portfolio optimization utilities
# ---------------------------------------------------------

def portfolio_variance(w, cov):
    """Portfolio variance: w' * Sigma * w"""
    return w @ cov @ w

def min_variance_portfolio(cov, allow_short=False):
    """
    Compute minimum variance portfolio weights.
    
    Parameters
    ----------
    cov : np.ndarray
        N x N covariance matrix
    allow_short : bool
        If False, constrain w >= 0
        
    Returns
    -------
    w : np.ndarray
        Optimal portfolio weights
    """
    n = cov.shape[0]
    w0 = np.ones(n) / n
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = None if allow_short else [(0, 1) for _ in range(n)]
    
    result = minimize(
        portfolio_variance, w0, args=(cov,), method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-10, 'maxiter': 1000}
    )
    
    return result.x if result.success else w0


def sample_covariance(returns_df):
    """Standard sample covariance (baseline)"""
    return returns_df.cov().values

In [ ]:
# ---------------------------------------------------------
# Rolling Backtest Engine
# ---------------------------------------------------------

# Backtest parameters
LOOKBACK = 252           # Days of history for estimation
REBALANCE_FREQ = 21      # Days between rebalances (~monthly)
MIN_HISTORY = 252        # Minimum days before starting

# Initialize results storage
dates = returns.index[MIN_HISTORY:]
n_assets = returns.shape[1]

# Track portfolio values
portfolio_values = {
    'Equal Weight': [1.0],
    'Sample Cov MVO': [1.0],
    'Your Estimator MVO': [1.0]
}

# Track weights for analysis
weights_history = {
    'Equal Weight': [],
    'Sample Cov MVO': [],
    'Your Estimator MVO': []
}

# Current weights
current_weights = {
    'Equal Weight': np.ones(n_assets) / n_assets,
    'Sample Cov MVO': np.ones(n_assets) / n_assets,
    'Your Estimator MVO': np.ones(n_assets) / n_assets
}

# Run backtest
rebalance_dates = []
for i, date in enumerate(dates):
    idx = returns.index.get_loc(date)
    
    # Rebalance check
    if i % REBALANCE_FREQ == 0:
        rebalance_dates.append(date)
        
        # Get lookback window
        lookback_returns = returns.iloc[idx-LOOKBACK:idx]
        
        # Equal weight — no estimation
        current_weights['Equal Weight'] = np.ones(n_assets) / n_assets
        
        # Sample covariance MVO
        cov_sample = sample_covariance(lookback_returns)
        current_weights['Sample Cov MVO'] = min_variance_portfolio(cov_sample)
        
        # Your estimator MVO
        cov_yours = estimate_covariance(lookback_returns)
        current_weights['Your Estimator MVO'] = min_variance_portfolio(cov_yours)
    
    # Store weights
    for name in current_weights:
        weights_history[name].append(current_weights[name].copy())
    
    # Compute daily returns
    daily_ret = returns.iloc[idx].values
    
    for name in portfolio_values:
        port_ret = current_weights[name] @ daily_ret
        portfolio_values[name].append(portfolio_values[name][-1] * (1 + port_ret))

# Convert to DataFrames
portfolio_df = pd.DataFrame(portfolio_values, index=[dates[0] - pd.Timedelta(days=1)] + list(dates))
print(f"Backtest complete: {len(rebalance_dates)} rebalance periods")

---
## §5 Results

In [ ]:
# ---------------------------------------------------------
# Plot: Cumulative Performance
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))

colors = {'Equal Weight': '#888888', 'Sample Cov MVO': '#2c5282', 'Your Estimator MVO': '#c53030'}
for name in portfolio_df.columns:
    ax.plot(portfolio_df.index, portfolio_df[name], label=name, 
            color=colors[name], linewidth=2 if 'Your' in name else 1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value ($1 initial)')
ax.set_title('Cumulative Performance: Your Estimator vs Baselines')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Performance Metrics Table
# ---------------------------------------------------------

def compute_metrics(values_series, rf=0.0):
    """Compute standard portfolio metrics"""
    returns = values_series.pct_change().dropna()
    
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - rf) / ann_vol if ann_vol > 0 else 0
    
    # Max drawdown
    cummax = values_series.cummax()
    drawdown = (values_series - cummax) / cummax
    max_dd = drawdown.min()
    
    return {
        'Ann. Return (%)': ann_ret * 100,
        'Ann. Volatility (%)': ann_vol * 100,
        'Sharpe Ratio': sharpe,
        'Max Drawdown (%)': max_dd * 100,
        'Final Value ($)': values_series.iloc[-1]
    }

metrics = pd.DataFrame({name: compute_metrics(portfolio_df[name]) 
                        for name in portfolio_df.columns}).T.round(2)

print("Performance Summary:\n")
display(metrics)

In [ ]:
# ---------------------------------------------------------
# Plot: Weight Concentration Over Time
# ---------------------------------------------------------

def herfindahl_index(weights):
    """HHI: sum of squared weights (lower = more diversified)"""
    return np.sum(weights ** 2)

# Compute HHI for each strategy over time
hhi_series = {name: [herfindahl_index(w) for w in weights_history[name]] 
              for name in weights_history}
hhi_df = pd.DataFrame(hhi_series, index=dates)

fig, ax = plt.subplots(figsize=(12, 4))
for name in hhi_df.columns:
    ax.plot(hhi_df.index, hhi_df[name], label=name, color=colors[name], alpha=0.8)

ax.axhline(1/n_assets, color='black', linestyle='--', linewidth=1, alpha=0.5, 
           label=f'Equal weight HHI = {1/n_assets:.3f}')
ax.set_xlabel('Date')
ax.set_ylabel('Herfindahl Index (HHI)')
ax.set_title('Portfolio Concentration Over Time')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Plot: Rolling Volatility Comparison
# ---------------------------------------------------------

ROLLING_WINDOW = 63  # ~3 months

portfolio_returns = portfolio_df.pct_change().dropna()
rolling_vol = portfolio_returns.rolling(ROLLING_WINDOW).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(12, 4))
for name in rolling_vol.columns:
    ax.plot(rolling_vol.index, rolling_vol[name], label=name, color=colors[name], alpha=0.8)

ax.set_xlabel('Date')
ax.set_ylabel('Rolling Volatility (%, annualized)')
ax.set_title(f'{ROLLING_WINDOW}-Day Rolling Volatility')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Analysis: Performance in Different Market Conditions
# ---------------------------------------------------------

# Define market regimes based on rolling volatility of equal-weight portfolio
eq_vol = portfolio_returns['Equal Weight'].rolling(63).std() * np.sqrt(252)
vol_threshold = eq_vol.quantile(0.7)

high_vol_mask = eq_vol > vol_threshold
low_vol_mask = eq_vol <= vol_threshold

print("Performance by Market Regime:\n")
print(f"High volatility periods: {high_vol_mask.sum()} days")
print(f"Low volatility periods: {low_vol_mask.sum()} days")

regime_metrics = {}
for regime, mask in [('High Vol', high_vol_mask), ('Low Vol', low_vol_mask)]:
    regime_returns = portfolio_returns[mask].dropna()
    if len(regime_returns) > 0:
        regime_metrics[regime] = {
            name: {
                'Ann. Return (%)': regime_returns[name].mean() * 252 * 100,
                'Ann. Volatility (%)': regime_returns[name].std() * np.sqrt(252) * 100,
            }
            for name in regime_returns.columns
        }

for regime, data in regime_metrics.items():
    print(f"\n{regime} Period:")
    display(pd.DataFrame(data).T.round(2))

In [ ]:
# ---------------------------------------------------------
# Final Weight Snapshot
# ---------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
tickers = returns.columns.tolist()

for ax, name in zip(axes, current_weights.keys()):
    w = current_weights[name]
    ax.bar(range(len(w)), w, color=colors[name], alpha=0.7)
    ax.set_xticks(range(len(w)))
    ax.set_xticklabels(tickers, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Weight')
    ax.set_title(f'{name}\nHHI = {herfindahl_index(w):.3f}')
    ax.set_ylim(0, max(0.3, w.max() * 1.1))

plt.suptitle('Final Portfolio Weights (Last Rebalance)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## §6 Interpretation

*Did the theory hold? When should someone use your estimator?*

[Write ~300 words analyzing your results. Address these questions:]

1. **Did your estimator beat the sample covariance baseline?** If yes, by how much? If no, why not?

2. **When did your estimator perform best vs worst?** Look at high-vol vs low-vol periods. Does this match what theory predicts?

3. **What happened to portfolio concentration (HHI)?** Did your estimator produce more or less diversified portfolios?

4. **What are the limitations?** When would you NOT recommend using this estimator?

5. **Key takeaway:** In one sentence, what did you learn?

---
<p style='text-align:center; color:#999; font-style:italic; font-size:0.9em;'>
Module [X] complete. See integration notebook for side-by-side comparison with other estimators.
</p>